## Data Collection Notebook

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Justification
</h3>

    I have chosen to compare education statistics from around England as there is major debate on the performance of students and quality of education recieved in the North vs. South of England. Therefore I will be collecting data on the retention and results across a range of 16-18 Studies (including A levels, AS levels, Tech  and Applied General Levels, Technical certificates and T Levels) which cover academic, vocational and technical options for UK Further Education (16-18 Studies).

    The data I am sourcing is unfortunately not available via API due to a lack of consistency in the Department of Education systems, however they do share it as Open Datasets (downloadable CSVs) along with a guide from the below site:
    - https://explore-education-statistics.service.gov.uk/data-catalogue

    I will be collecting data on the performance of students based on various personal characteristics (Gender, Ethnicity etc.) and rurality of the area. Also I will collect overall results across all subject and a more dedicated dataset for STEM subject combinations. Lastly, I will collect the data for retention (students who return to school from GCSEs - 14-16 Studies - for Further Education).

    My data is limited by the time-periods available to me which was the last 5 academic years from 2019-20 until 2023-24 meaning trends over time will not be my main point of analysis. Instead I will focus more on comparing regions and the patterns that emerge from a simplistic "geographical" point of view.

</div>

In [85]:
## Imports

import os

import pandas as pd

import sys
sys.path.append('../..')

from final_project.scripts.data_collection_utils import set_nulls, adjust_time_periods

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Reading CSV from EES - Explore Education Statistics - Data Access point for Department of Education

</h3>


</div>

In [3]:
rurality_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+
                                             "/data/raw/EES_attainment_by_rurality.csv"))

chars_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+
                                          "/data/raw/EES_attainment_by_characteristics.csv"))

retention_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+
                                              "/data/raw/EES_retention_by_region.csv"))

results_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+
                                            "/data/raw/EES_results_by_subject.csv"))

stem_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+
                                         "/data/raw/EES_stem_by_sex.csv"))

In [4]:
rurality_df = pd.read_csv(
    "https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/285687fd-7a12-4cbc-8a93-2fccfa0df053/csv")

chars_df = pd.read_csv(
    "https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/ca6f1e5a-35a5-4090-baf2-5611e43a7901/csv")

retention_df = pd.read_csv(
    "https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/1fece430-ce67-4027-ad93-068a4c8e1d5a/csv")

## low_memory=False is required to avoid warning that the count of students per grade (numerical) have non-numerical entries
## which is due to custom nulls defined by DfE to show different reasons for missing data
results_df = pd.read_csv(
    "https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/10aeadde-eece-4e45-a9b8-8b6c2fbd5124/csv",
    low_memory=False)

stem_df = pd.read_csv(
    "https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/f46bb916-d67c-403f-8e01-586ae5c15e61/csv")

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Sorting out Rurality Data

</h3>

As evidenced from the below table, the rurality data accounts for multiple types of Urban and Rural Areas, each of which has a certain number of students, the Unknown Rurality rows are obsolete as they contain no students anyway so these should be removed. It is worth noting here that while the table shows an overwhelming proportion of students attend schools in Urban Areas compared to Rural Areas, I do not yet know the proportions across or within regions. In the case that a certain region has significant proportion of a students in Rural and Urban Areas, this will allow for fair analysis.

Therefore I will be:
- pulling the important/ useful columns only
- deal with the custom null values "z", "x" and "c" and adjusting the time_period column to something more readable
- identify actual null values and determine validity

</div>

In [5]:
rurality_df.groupby("rurality_name").agg({"number_of_students_potential": "sum"})

,number_of_students_potential
rurality_name,
Rural - Hamlet and isolated dwellings,57615
Rural - Town and fringe,88090
Rural - Village,29367
Unknown rurality,0
Urban - City and town,1432430
Urban - Major conurbation,1036029
Urban - Minor conurbation,97582


In [6]:
rurality_df = rurality_df[["time_period", "region_name", "rurality_name", "number_of_students_level3",
    "number_of_students_alev", "number_of_students_acad", "number_of_students_agen", "number_of_students_highest_entry_was_l2",
    "number_of_students_tlev", "number_of_students_technicalcertificate", "number_of_students_potential",
    "pc_achieving_3_astar_to_a_alev", "pc_achieving_atleast_two_alev", "aps_per_entry_grade_alev"]]

In [7]:
rurality_df = rurality_df[~(rurality_df["rurality_name"] == "Unknown rurality")].reset_index(drop=True)
# 9 regions *5 time periods  *6 ruralities = 270 rows

rurality_df = rurality_df.map(set_nulls)

rurality_df["time_period"] = rurality_df["time_period"].apply(adjust_time_periods)

In [8]:
rurality_df["pc_achieving_alev"] = ((rurality_df["number_of_students_alev"] / rurality_df["number_of_students_potential"]) *100).round(3)

rurality_df.drop(columns=["number_of_students_level3", "number_of_students_acad", "number_of_students_agen",
                      "number_of_students_highest_entry_was_l2", "number_of_students_tlev", "number_of_students_technicalcertificate"],
                      inplace=True)

In [9]:
rurality_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   time_period                     270 non-null    object 
 1   region_name                     270 non-null    object 
 2   rurality_name                   270 non-null    object 
 3   number_of_students_alev         270 non-null    int64  
 4   number_of_students_potential    270 non-null    int64  
 5   pc_achieving_3_astar_to_a_alev  214 non-null    float64
 6   pc_achieving_atleast_two_alev   215 non-null    float64
 7   aps_per_entry_grade_alev        215 non-null    object 
 8   pc_achieving_alev               227 non-null    float64
dtypes: float64(3), int64(2), object(4)
memory usage: 19.1+ KB


In [11]:
rurality_df.groupby("region_name")[["pc_achieving_3_astar_to_a_alev",
                                   "pc_achieving_atleast_two_alev",
                                   "aps_per_entry_grade_alev",
                                   "pc_achieving_alev"]].apply(lambda x: x.isnull().sum(), include_groups=False)

,pc_achieving_3_astar_to_a_alev,pc_achieving_atleast_two_alev,aps_per_entry_grade_alev,pc_achieving_alev
region_name,,,,
East Midlands,0,0,0,0
East of England,5,5,5,5
London,11,10,10,6
North East,10,10,10,5
North West,5,5,5,5
South East,5,5,5,5
South West,10,10,10,10
West Midlands,10,10,10,7
Yorkshire and The Humber,0,0,0,0


In [12]:
rurality_df.groupby("time_period")[["pc_achieving_3_astar_to_a_alev",
                                   "pc_achieving_atleast_two_alev",
                                   "aps_per_entry_grade_alev",
                                   "pc_achieving_alev"]].apply(lambda x: x.isnull().sum(), include_groups=False)

,pc_achieving_3_astar_to_a_alev,pc_achieving_atleast_two_alev,aps_per_entry_grade_alev,pc_achieving_alev
time_period,,,,
2019 to 20,12,12,12,8
2020 to 21,12,11,11,9
2021 to 22,10,10,10,9
2022 to 23,11,11,11,9
2023 to 24,11,11,11,8


<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Validity    
</h3>

While there are some null values, these will not interfere with sums/ averages as they are purely a result of 0 students attending that particular type of Area (Rural or Urban options) in that region/ academic year. Since there is no systematic issue here there is no need to parse out and remove these explicity.

The data can now be saved to local csv in ../data/raw

</div>

In [13]:
with open(rurality_path, "w") as f:
    rurality_df.to_csv(f, index=False)

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Sorting out Characteristics Data

</h3>
    
From the below table it is clear that many different types of characteristics are recorded in this dataset, I will only require Total Students for a total count, Disadvantage, Ethnicity and Sex as will be the clearest and most meaningul from reader's point of view. These are common topics of conversation for UK education so I think it will be important to compare the performance and capabilities within the different cohorts of each characteristic type.

Therefore I will be:
- pulling the important/ useful columns only
- deal with the custom null values "z", "x" and "c" and adjusting the time_period column to something more readable
- identify actual null values and determine validity
    
</div>

In [14]:
chars_df.groupby("characteristic_type").agg({"characteristic_value": "unique"})

,characteristic_value
characteristic_type,
All Students,[All state-funded students]
Disadvantage,"[Disadvantaged, Non-Disadvantaged, Unknown dis..."
Ethnicity,"[Black or Black British, Mixed Dual background..."
FSM,"[Eligible for FSM, Not eligible for FSM, Unkno..."
First Language,"[English language, Other than English language..."
Prior Attainment,"[Priors 0 to < 4, Priors 4 to < 7, Priors 7+, ..."
SEN Provision,"[Total EHC plans and statements of SEN, Total ..."
Sex,"[Female, Male]"


In [15]:
chars_df = chars_df[["time_period", "region_name", "geographic_level", "characteristic_type", "characteristic_value",
    "number_of_students_level3", "number_of_students_alev", "number_of_students_acad", "number_of_students_agen",
    "number_of_students_highest_entry_was_l2", "number_of_students_tlev", "number_of_students_technicalcertificate",
    "number_of_students_potential", "pc_achieving_3_astar_to_a_alev", "pc_achieving_atleast_two_alev", "aps_per_entry_grade_alev"]]

In [16]:
chars_df = chars_df[(chars_df["geographic_level"] == "Regional") &
                    (chars_df["characteristic_type"].isin(["Disadvantage", "Ethnicity", "Sex", "All Students"]))].reset_index(drop=True)
# 9 regions *5 time periods  12* characteristics = 540 rows

chars_df = chars_df.map(set_nulls)

chars_df["time_period"] = chars_df["time_period"].apply(adjust_time_periods)

In [17]:
chars_df["pc_achieving_alev"] = ((chars_df["number_of_students_alev"] / chars_df["number_of_students_potential"]) *100).round(3)

chars_df.drop(columns=["geographic_level", "number_of_students_level3", "number_of_students_acad", "number_of_students_agen",
                      "number_of_students_highest_entry_was_l2", "number_of_students_tlev", "number_of_students_technicalcertificate"],
                      inplace=True)

In [18]:
chars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 540 entries, 0 to 539
Data columns (total 10 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   time_period                     540 non-null    object 
 1   region_name                     540 non-null    object 
 2   characteristic_type             540 non-null    object 
 3   characteristic_value            540 non-null    object 
 4   number_of_students_alev         540 non-null    int64  
 5   number_of_students_potential    540 non-null    int64  
 6   pc_achieving_3_astar_to_a_alev  540 non-null    float64
 7   pc_achieving_atleast_two_alev   540 non-null    float64
 8   aps_per_entry_grade_alev        540 non-null    object 
 9   pc_achieving_alev               540 non-null    float64
dtypes: float64(3), int64(2), object(5)
memory usage: 42.3+ KB


<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Validity    
</h3>

There are no null values in this particular dataset so the data can now be saved to local csv in ../data/raw

</div>

In [27]:
with open(chars_path, "w") as f:
    chars_df.to_csv(f, index=False)

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Sorting out Retention Data

</h3>

Seeing as the rows for cohort type are repeated more than (5 time_period * 9 regions =) 45 times this indicated a dataset where the values have been given per town/city, I will need to filter where the geographical_level is Regional only

Therefore I will be:
- pulling the important/ useful columns only
- deal with the custom null values "z", "x" and "c" and adjusting the time_period column to something more readable
- identify actual null values and determine validity
    
</div>

In [43]:
retention_df.groupby("exam_cohort").agg({"student_count_year_1": ["count", "mean"]})

student_count_year_1             
                                     count         mean
exam_cohort                                            
A level                                809  2858.627936
Academic                               809  2884.751545
Applied general                        809  1090.687268
Tech level                             809   347.245983
Technical certificate                  809   110.796044

In [29]:
retention_df = retention_df[["time_period", "region_name", "geographic_level", "exam_cohort", "student_count_year_1",
    "student_count_year_2", "retained", "retained_and_assessed", "returned_and_retained", "perc_retained",
    "perc_retained_and_assessed", "perc_returned_and_retained"]]

In [44]:
retention_df = retention_df[retention_df["geographic_level"] == "Regional"].reset_index(drop=True)
# 9 regions *5 cohorts *5 time period = 225 rows

retention_df = retention_df.map(set_nulls)

retention_df["time_period"] = retention_df["time_period"].apply(adjust_time_periods)

In [45]:
retention_df.drop(columns=["geographic_level"], inplace=True)

In [46]:
retention_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225 entries, 0 to 224
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   time_period                 225 non-null    object 
 1   region_name                 225 non-null    object 
 2   exam_cohort                 225 non-null    object 
 3   student_count_year_1        225 non-null    int64  
 4   student_count_year_2        180 non-null    float64
 5   retained                    225 non-null    int64  
 6   retained_and_assessed       225 non-null    int64  
 7   returned_and_retained       180 non-null    float64
 8   perc_retained               225 non-null    float64
 9   perc_retained_and_assessed  225 non-null    float64
 10  perc_returned_and_retained  180 non-null    float64
dtypes: float64(5), int64(3), object(3)
memory usage: 19.5+ KB


In [51]:
retention_df.groupby("exam_cohort")[["student_count_year_2",
                                                    "returned_and_retained",
                                                    "perc_returned_and_retained"]].apply(lambda x: x.isnull().sum(), include_groups=False)

,student_count_year_2,returned_and_retained,perc_returned_and_retained
exam_cohort,,,
A level,0,0,0
Academic,0,0,0
Applied general,0,0,0
Tech level,0,0,0
Technical certificate,45,45,45


<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Validity    
</h3>

The null values systematically appear for the Technical Certificate cohort when calculating students "returned". As explained by the DfE online information, Technical Certifications are the only 16-18 Study recorded that are of Level 2 (finish in the first year) and therefore do not have students "returning" for the second year (as opposed to being "retained" from 14-16 Studies which they are). Therefore this is a meaningful implementation of nulls which I will not interfere with.

The data can now be saved to local csv in ../data/raw

</div>

In [52]:
with open(retention_path, "w") as f:
    retention_df.to_csv(f, index=False)

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Sorting out Results Data

</h3>

The following "groupby" demonstrates that there are 2 recorded qualifications, both of which I will keep as I am not yet certain of my analysis goals/ steps etc. Also I can be seen that each Suject Area has 1 or more Subject within it therefore both of these columns are necesary to encapsulate all the students.

Therefore I will be:
- pulling the important/ useful columns only
- deal with the custom null values "z", "x" and "c" and adjusting the time_period column to something more readable
- identify actual null values and determine validity
    
</div>

In [53]:
results_df.groupby(["qualification", "subject_area"]).agg({"subject_name": ["nunique", "unique"]})

subject_name  \
                                             nunique   
qualification subject_area                             
A level       Accounting and Finance               1   
              All STEM subjects                    1   
              All facilitating subjects            1   
              All subjects                         1   
              Anthropology                         1   
...                                              ...   
AS level      Science - Biology                    1   
              Science - Chemistry                  1   
              Science - Physics                    1   
              Science - other                      6   
              Sociology                            1   

                                                                                            
                                                                                    unique  
qualification subject_area                                                                  
A level       Accounting and Finance                        [Total Accounting and finance]  
              All STEM subjects                                      [Total STEM subjects]  
              All facilitating subjects                      [Total facilitating subjects]  
              All subjects                                                [Total subjects]  
              Anthropology                                            [Total Anthropology]  
...                                                                                    ...  
AS level      Science - Biology                                            [Total Biology]  
              Science - Chemistry                                        [Total Chemistry]  
              Science - Physics                                            [Total Physics]  
              Science - other            [Science in Society, Science SA, Environmental...  
              Sociology                                                  [Total Sociology]  

[74 rows x 2 columns]

In [54]:
results_df = results_df[["time_period", "geographic_level", "region_name", "qualification", "subject_area", 
    "subject_name", "entry_count", "perc_astar_grade_achieved", "perc_astar_a_grade_achieved",
    "perc_astar_b_grade_achieved", "perc_astar_c_grade_achieved", "perc_astar_d_grade_achieved",
    "perc_astar_e_grade_achieved"]]

In [55]:
results_df = results_df[results_df["geographic_level"] == "Regional"].reset_index(drop=True)
# 9 regions *2 qualifications *5 time period *109 subjects = 9810 rows

results_df = results_df.map(set_nulls)

results_df["time_period"] = results_df["time_period"].apply(adjust_time_periods)

In [ ]:
results_df.drop(columns=["geographic_level"], inplace=True)

results_df = results_df[~results_df["entry_count"].isna()].reset_index(drop=True)
# remove discontinued subjects as they are not useful for analysis

In [57]:
results_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8847 entries, 0 to 8846
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   time_period                  8847 non-null   object 
 1   region_name                  8847 non-null   object 
 2   qualification                8847 non-null   object 
 3   subject_area                 8847 non-null   object 
 4   subject_name                 8847 non-null   object 
 5   entry_count                  8847 non-null   float64
 6   perc_astar_grade_achieved    5737 non-null   float64
 7   perc_astar_a_grade_achieved  5737 non-null   float64
 8   perc_astar_b_grade_achieved  5737 non-null   float64
 9   perc_astar_c_grade_achieved  5737 non-null   float64
 10  perc_astar_d_grade_achieved  5737 non-null   float64
 11  perc_astar_e_grade_achieved  5737 non-null   float64
dtypes: float64(7), object(5)
memory usage: 829.5+ KB


In [75]:
results_df.groupby(["qualification", "entry_count"])[
    ["perc_astar_grade_achieved",
     "perc_astar_a_grade_achieved"]].apply(lambda x: x.isnull().sum(),
                                           include_groups=False).sort_values("entry_count")

perc_astar_grade_achieved  \
qualification entry_count                              
A level       0.0                               1102   
AS level      0.0                               2008   
              1.0                                  0   
A level       1.0                                  0   
              2.0                                  0   
...                                              ...   
              125086.0                             0   
              125975.0                             0   
              128567.0                             0   
              130250.0                             0   
              135039.0                             0   

                           perc_astar_a_grade_achieved  
qualification entry_count                               
A level       0.0                                 1102  
AS level      0.0                                 2008  
              1.0                                    0  
A level       1.0                                    0  
              2.0                                    0  
...                                                ...  
              125086.0                               0  
              125975.0                               0  
              128567.0                               0  
              130250.0                               0  
              135039.0                               0  

[2488 rows x 2 columns]

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Validity    
</h3>

Evidently the null values occur only when the number of students entered into the subject is 0, therefore this is meaningful and should not be removed.

The data can now be saved to local csv in ../data/raw

</div>

In [76]:
with open(results_path, "w") as f:
    results_df.to_csv(f, index=False)

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Sorting out STEM Data

</h3>

From the table below we can see that the STEM subjects are presented in every possible combination and there are some obsolete values that can be removed such as the 0 combination and All Students options which will likely cause duplications when counting. Geographic level also needs to be set to Regional to maintain the scale of analysis identified earlier.

Therefore I will be:
- pulling the important/ useful columns only
- deal with the custom null values "z", "x" and "c" and adjusting the time_period column to something more readable
- identify actual null values and determine validity
    
</div>

In [79]:
stem_df.groupby("num_maths_science").agg({"sub_name_comb": ["nunique", "unique"]})

sub_name_comb  \
                        nunique   
num_maths_science                 
0                             1   
1                             7   
2                            16   
3                            21   
4                            16   
5                             7   
6                             2   
Total Students                1   

                                                                      
                                                              unique  
num_maths_science                                                     
0                                      [Zero Maths/Science subjects]  
1                  [Chemistry, One Maths/Science subject, Physics...  
2                  [Maths & Chemistry, Further Maths & Chemistry,...  
3                  [Further Maths & Biology & Physics, Biology & ...  
4                  [Maths & Further Maths & Biology & Chemistry, ...  
5                  [Maths & Further Maths & Biology & Chemistry &...  
6                  [Maths & Further Maths & Biology & Chemistry &...  
Total Students                                      [Total Students]

In [80]:
stem_df = stem_df[["time_period", "geographic_level", "region_name", "characteristic_sex", "sub_name_comb",
    "num_maths_science", "num_entered_comb_and_no_other_matsci", "perc_entered_comb_and_no_other_matsci"]]

In [81]:
stem_df = stem_df[stem_df["geographic_level"] == "Regional"].reset_index(drop=True)
# 9 regions *3 characteristics *5 time periods *71 subjects = 9585 rows

stem_df = stem_df.map(set_nulls)

stem_df["time_period"] = stem_df["time_period"].apply(adjust_time_periods)

In [82]:
stem_df.drop(columns=["geographic_level"], inplace=True)

stem_df = stem_df[~(stem_df["num_maths_science"] == "Total Students")].reset_index(drop=True)
# 9 regions *3 sexes (due to Null) *5 time periods *70 subjects = 9450 rows

# Remove rows where sub_name_comb contains "one", "two" etc. as these will cause double/ triple counting when eg. taking sums
stem_df = stem_df[~stem_df["sub_name_comb"].str.contains("Zero|One|Two|Three|Four|Five|Six",
                                                         case=False, na=False)].reset_index(drop=True)
# 9 regions *3 sexes (due to Null) *5 time periods *63 subjects = 8505 rows

In [83]:
stem_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8505 entries, 0 to 8504
Data columns (total 7 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   time_period                            8505 non-null   object 
 1   region_name                            8505 non-null   object 
 2   characteristic_sex                     8505 non-null   object 
 3   sub_name_comb                          8505 non-null   object 
 4   num_maths_science                      8505 non-null   object 
 5   num_entered_comb_and_no_other_matsci   8505 non-null   int64  
 6   perc_entered_comb_and_no_other_matsci  8505 non-null   float64
dtypes: float64(1), int64(1), object(5)
memory usage: 465.2+ KB


<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Validity    
</h3>

There are no null values in this particular dataset so the data can now be saved to local csv in ../data/raw

</div>

In [84]:
with open(stem_path, "w") as f:
    stem_df.to_csv(f, index=False)

<div style="padding: 0px 0px 0px 20px; border-left: 8px solid #ED9255; border-radius: 8px">
<h3>
    Collection Summary    
</h3>

Data from 5 different datasets have been collected, cleaned & filtered through and stored into ../data/raw. This data will be inserted into the database designed in the next section - NB02 Data Processing.

</div>